# Dataset Inventory

## Objective

The goal of this notebook is to understand the structure of the FlyRank internship warehouse before selecting a research direction.

We will inspect the available tables, their schemas, row counts, and relationships between them.

In [50]:
from huggingface_hub import hf_hub_download
import duckdb

local_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_clients.parquet",
    repo_type="dataset"
)

print("Downloaded to:", local_file)

con = duckdb.connect()

client_schema = con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(?)
""", [local_file]).fetchdf()

client_schema


Downloaded to: C:\Users\anasm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\dim_clients.parquet


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


In [51]:
client_schema

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


In [52]:
from huggingface_hub import hf_hub_download

content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset"
)

content_schema = con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(?)
""", [content_file]).fetchdf()

content_schema

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [53]:
performance_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset"
)

performance_schema = con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(?)
""", [performance_file]).fetchdf()

performance_schema


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [54]:
query_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_query_90d.parquet",
    repo_type="dataset"
)

query_schema = con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(?)
""", [query_file]).fetchdf()

query_schema


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None


In [55]:
content_stats = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(content_hash_id) AS content_id_count,
        COUNT(DISTINCT content_hash_id) AS unique_content_id_count
    FROM read_parquet(?)
""", [content_file]).fetchdf()

content_stats

,row_count,content_id_count,unique_content_id_count
0,519606,519606,519606


In [56]:
client_stats = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(client_hash_id) AS client_id_count,
        COUNT(DISTINCT client_hash_id) AS unique_client_id_count
    FROM read_parquet(?)
""", [local_file]).fetchdf()

client_stats

,row_count,client_id_count,unique_client_id_count
0,104,104,104


In [57]:
performance_stats = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(report_date) AS date_count,
        COUNT(client_hash_id) AS client_id_count,
        COUNT(content_hash_id) AS content_id_count,
        COUNT(DISTINCT report_date) AS unique_dates,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_contents
    FROM read_parquet(?)
""", [performance_file]).fetchdf()

performance_stats

,row_count,date_count,client_id_count,content_id_count,unique_dates,unique_clients,unique_contents
0,11694072,11694072,11694072,11694072,30,65,409205


In [58]:
duplicate_grain = con.execute("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet(?)
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""", [performance_file]).fetchdf()

duplicate_grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count
0,2026-06-20,client_a22068e339bf95f5,content_0c1759db98b39c91,2
1,2026-06-17,client_1a8bf67cad4ee525,content_23bd1640d6b9d83f,2
2,2026-06-24,client_4a18d1793d92fb84,content_4753dfdc4e7ea44e,2
3,2026-06-16,client_b77d0d5f08f05e64,content_41a620c650d2fea1,2
4,2026-06-25,client_b77d0d5f08f05e64,content_3d9fc8e76439c31e,2
5,2026-06-27,client_810019792c9b8efc,content_9c617d81ef844bd6,2
6,2026-06-28,client_b77d0d5f08f05e64,content_2c0b9473540236fc,2
7,2026-06-28,client_b77d0d5f08f05e64,content_d9d95aebc5e40e25,2
8,2026-06-28,client_b77d0d5f08f05e64,content_99a3e159bf3e5306,2
9,2026-06-24,client_def0955f7a377868,content_0a91140fa7c2cbd4,2


In [59]:
duplicate_rows = con.execute("""
    SELECT *
    FROM read_parquet(?)
    WHERE report_date = '2026-06-28'
      AND client_hash_id = 'client_810019792c9b8efc'
      AND content_hash_id = 'content_0064be1867cf7ae8'
""", [performance_file]).fetchdf()

duplicate_rows

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-28,client_810019792c9b8efc,content_0064be1867cf7ae8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-28,client_810019792c9b8efc,content_0064be1867cf7ae8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [60]:
duplicate_check = con.execute("""
    SELECT COUNT(*) AS unique_rows
    FROM (
        SELECT DISTINCT *
        FROM read_parquet(?)
        WHERE report_date = '2026-06-28'
          AND client_hash_id = 'client_810019792c9b8efc'
          AND content_hash_id = 'content_0064be1867cf7ae8'
    )
""", [performance_file]).fetchdf()

duplicate_check

,unique_rows
0,1


In [61]:
duplicate_summary = con.execute("""
    SELECT
        COUNT(*) AS total_groups,
        SUM(CASE WHEN row_count > 1 THEN 1 ELSE 0 END) AS duplicate_groups,
        SUM(row_count - 1) AS duplicate_rows
    FROM (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            COUNT(*) AS row_count
        FROM read_parquet(?)
        GROUP BY
            report_date,
            client_hash_id,
            content_hash_id
    )
""", [performance_file]).fetchdf()

duplicate_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_groups,duplicate_groups,duplicate_rows
0,11687682,6390.0,6390.0


In [62]:
query_grain = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_contents,
        COUNT(DISTINCT query_hash_id) AS unique_queries
    FROM read_parquet(?)
""", [query_file]).fetchdf()

query_grain

,total_rows,unique_clients,unique_contents,unique_queries
0,2414248,52,133852,1180090


In [63]:
query_duplicate_check = con.execute("""
    SELECT
        COUNT(*) AS total_groups,
        SUM(CASE WHEN row_count > 1 THEN 1 ELSE 0 END) AS duplicate_groups,
        SUM(row_count - 1) AS duplicate_rows
    FROM (
        SELECT
            client_hash_id,
            content_hash_id,
            query_hash_id,
            COUNT(*) AS row_count
        FROM read_parquet(?)
        GROUP BY
            client_hash_id,
            content_hash_id,
            query_hash_id
    )
""", [query_file]).fetchdf()

query_duplicate_check


,total_groups,duplicate_groups,duplicate_rows
0,2414248,0.0,0.0


In [64]:
client_relationship = con.execute("""
    SELECT
        COUNT(DISTINCT c.client_hash_id) AS content_clients,
        COUNT(DISTINCT d.client_hash_id) AS matched_clients
    FROM read_parquet(?) c
    LEFT JOIN read_parquet(?) d
        ON c.client_hash_id = d.client_hash_id
    WHERE d.client_hash_id IS NOT NULL
""", [content_file, local_file]).fetchdf()

client_relationship

,content_clients,matched_clients
0,84,84


In [65]:
content_relationship = con.execute("""
    SELECT
        COUNT(DISTINCT p.content_hash_id) AS performance_contents,
        COUNT(DISTINCT c.content_hash_id) AS matched_contents
    FROM read_parquet(?) p
    LEFT JOIN read_parquet(?) c
        ON p.content_hash_id = c.content_hash_id
    WHERE c.content_hash_id IS NOT NULL
""", [performance_file, content_file]).fetchdf()

content_relationship

,performance_contents,matched_contents
0,409205,409205


In [66]:
query_content_relationship = con.execute("""
    SELECT
        COUNT(DISTINCT q.content_hash_id) AS query_contents,
        COUNT(DISTINCT c.content_hash_id) AS matched_contents
    FROM read_parquet(?) q
    LEFT JOIN read_parquet(?) c
        ON q.content_hash_id = c.content_hash_id
    WHERE c.content_hash_id IS NOT NULL
""", [query_file, content_file]).fetchdf()

query_content_relationship

,query_contents,matched_contents
0,133852,133852


In [67]:
query_client_relationship = con.execute("""
    SELECT
        COUNT(DISTINCT q.client_hash_id) AS query_clients,
        COUNT(DISTINCT c.client_hash_id) AS matched_clients
    FROM read_parquet(?) q
    LEFT JOIN read_parquet(?) c
        ON q.client_hash_id = c.client_hash_id
    WHERE c.client_hash_id IS NOT NULL
""", [query_file, local_file]).fetchdf()

query_client_relationship

,query_clients,matched_clients
0,52,52


                    dim_clients
                         │
                   client_hash_id
                         │
                         ▼
                    dim_content
                  ┌──────┴──────┐
                  │             │
          content_hash_id   content_hash_id
                  │             │
                  ▼             ▼
        daily_performance    query_90d
          date + client      client + content
          + content           + query

In [68]:
content_missing = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(client_hash_id) AS missing_client_hash_id,
        COUNT(*) - COUNT(content_hash_id) AS missing_content_hash_id,
        COUNT(*) - COUNT(keyword_hash_id) AS missing_keyword_hash_id,
        COUNT(*) - COUNT(content_type) AS missing_content_type,
        COUNT(*) - COUNT(search_volume) AS missing_search_volume,
        COUNT(*) - COUNT(main_intent) AS missing_main_intent,
        COUNT(*) - COUNT(word_count) AS missing_word_count,
        COUNT(*) - COUNT(last_optimized_date) AS missing_last_optimized_date
    FROM read_parquet(?)
""", [content_file]).fetchdf()

content_missing

,total_rows,missing_client_hash_id,missing_content_hash_id,missing_keyword_hash_id,missing_content_type,missing_search_volume,missing_main_intent,missing_word_count,missing_last_optimized_date
0,519606,0,0,71998,0,142622,148398,177768,474210


In [69]:
content_missing_all = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) - COUNT(client_hash_id) AS missing_client_hash_id,
        COUNT(*) - COUNT(content_hash_id) AS missing_content_hash_id,
        COUNT(*) - COUNT(keyword_hash_id) AS missing_keyword_hash_id,
        COUNT(*) - COUNT(url_hash_id) AS missing_url_hash_id,

        COUNT(*) - COUNT(keyword_char_count) AS missing_keyword_char_count,
        COUNT(*) - COUNT(keyword_token_count) AS missing_keyword_token_count,
        COUNT(*) - COUNT(url_char_count) AS missing_url_char_count,

        COUNT(*) - COUNT(content_created_date) AS missing_content_created_date,
        COUNT(*) - COUNT(content_updated_date) AS missing_content_updated_date,
        COUNT(*) - COUNT(content_type) AS missing_content_type,

        COUNT(*) - COUNT(search_volume) AS missing_search_volume,
        COUNT(*) - COUNT(competition) AS missing_competition,
        COUNT(*) - COUNT(competition_level) AS missing_competition_level,
        COUNT(*) - COUNT(cpc) AS missing_cpc,
        COUNT(*) - COUNT(main_intent) AS missing_main_intent,

        COUNT(*) - COUNT(backlinks) AS missing_backlinks,
        COUNT(*) - COUNT(category_count) AS missing_category_count,

        COUNT(*) - COUNT(keyword_created_date) AS missing_keyword_created_date,
        COUNT(*) - COUNT(provider_used) AS missing_provider_used,
        COUNT(*) - COUNT(model_used) AS missing_model_used,

        COUNT(*) - COUNT(char_count) AS missing_char_count,
        COUNT(*) - COUNT(word_count) AS missing_word_count,

        COUNT(*) - COUNT(last_optimized_date) AS missing_last_optimized_date,
        COUNT(*) - COUNT(optimization_eligible_date) AS missing_optimization_eligible_date,

        COUNT(*) - COUNT(is_published) AS missing_is_published,
        COUNT(*) - COUNT(is_deleted) AS missing_is_deleted

    FROM read_parquet(?)
""", [content_file]).fetchdf()

content_missing_all

,total_rows,missing_client_hash_id,missing_content_hash_id,missing_keyword_hash_id,missing_url_hash_id,missing_keyword_char_count,missing_keyword_token_count,missing_url_char_count,missing_content_created_date,missing_content_updated_date,...,missing_category_count,missing_keyword_created_date,missing_provider_used,missing_model_used,missing_char_count,missing_word_count,missing_last_optimized_date,missing_optimization_eligible_date,missing_is_published,missing_is_deleted
0,519606,0,0,71998,6525,0,0,0,0,0,...,0,71998,369936,84963,177768,177768,474210,474210,0,0


In [70]:
keyword_missing_pattern = con.execute("""
    SELECT
        COUNT(*) AS total_missing_keyword,
        COUNT(*) FILTER (
            WHERE keyword_created_date IS NULL
        ) AS also_missing_keyword_date,
        COUNT(*) FILTER (
            WHERE keyword_created_date IS NOT NULL
        ) AS keyword_date_available
    FROM read_parquet(?)
    WHERE keyword_hash_id IS NULL
""", [content_file]).fetchdf()

keyword_missing_pattern

,total_missing_keyword,also_missing_keyword_date,keyword_date_available
0,71998,71998,0


In [71]:
keyword_url_pattern = con.execute("""
    SELECT
        COUNT(*) AS total_missing_keyword,
        COUNT(*) FILTER (
            WHERE url_hash_id IS NULL
        ) AS also_missing_url,
        COUNT(*) FILTER (
            WHERE url_hash_id IS NOT NULL
        ) AS url_available
    FROM read_parquet(?)
    WHERE keyword_hash_id IS NULL
""", [content_file]).fetchdf()

keyword_url_pattern

,total_missing_keyword,also_missing_url,url_available
0,71998,5151,66847


In [72]:
content_text_pattern = con.execute("""
    SELECT
        COUNT(*) AS total_missing_char_count,
        COUNT(*) FILTER (
            WHERE word_count IS NULL
        ) AS also_missing_word_count,
        COUNT(*) FILTER (
            WHERE word_count IS NOT NULL
        ) AS word_count_available
    FROM read_parquet(?)
    WHERE char_count IS NULL
""", [content_file]).fetchdf()

content_text_pattern

,total_missing_char_count,also_missing_word_count,word_count_available
0,177768,177768,0


In [73]:
optimization_pattern = con.execute("""
    SELECT
        COUNT(*) AS total_missing_last_optimized,
        COUNT(*) FILTER (
            WHERE optimization_eligible_date IS NULL
        ) AS also_missing_eligible,
        COUNT(*) FILTER (
            WHERE optimization_eligible_date IS NOT NULL
        ) AS eligible_available
    FROM read_parquet(?)
    WHERE last_optimized_date IS NULL
""", [content_file]).fetchdf()

optimization_pattern

,total_missing_last_optimized,also_missing_eligible,eligible_available
0,474210,474210,0


In [74]:
performance_availability = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS NULL
        ) AS missing_gsc_available,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS NULL
        ) AS missing_ga4_available,

        COUNT(*) FILTER (
            WHERE client_has_gsc IS NULL
        ) AS missing_client_has_gsc,

        COUNT(*) FILTER (
            WHERE client_has_ga4 IS NULL
        ) AS missing_client_has_ga4
    FROM read_parquet(?)
""", [performance_file]).fetchdf()

performance_availability

,total_rows,missing_gsc_available,missing_ga4_available,missing_client_has_gsc,missing_client_has_ga4
0,11694072,0,2397428,0,0


In [75]:
query_grain = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_contents,
        COUNT(DISTINCT query_hash_id) AS unique_queries
    FROM read_parquet(?)
""", [query_file]).fetchdf()

query_grain

,total_rows,unique_clients,unique_contents,unique_queries
0,2414248,52,133852,1180090


In [76]:
ga4_availability_pattern = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE ga4_data_available IS NULL
        ) AS missing_ga4_available,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS NULL
            AND ga4_sessions IS NULL
        ) AS missing_flag_and_sessions,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS NULL
            AND ga4_sessions IS NOT NULL
        ) AS sessions_available,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS NOT NULL
            AND ga4_sessions IS NULL
        ) AS sessions_missing_with_flag
    FROM read_parquet(?)
""", [performance_file]).fetchdf()

ga4_availability_pattern

,missing_ga4_available,missing_flag_and_sessions,sessions_available,sessions_missing_with_flag
0,2397428,2397428,0,0


In [77]:
ga4_metrics_pattern = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE ga4_data_available IS NULL
        ) AS missing_ga4_available,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS NULL
            AND ga4_pageviews IS NULL
        ) AS missing_pageviews,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS NULL
            AND ga4_users IS NULL
        ) AS missing_users,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS NULL
            AND ga4_engaged_sessions IS NULL
        ) AS missing_engaged_sessions,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS NULL
            AND sessions_organic IS NULL
        ) AS missing_organic_sessions,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS NULL
            AND sessions_ai IS NULL
        ) AS missing_ai_sessions
    FROM read_parquet(?)
""", [performance_file]).fetchdf()

ga4_metrics_pattern

,missing_ga4_available,missing_pageviews,missing_users,missing_engaged_sessions,missing_organic_sessions,missing_ai_sessions
0,2397428,2397428,2397428,2397428,2397428,2397428


In [78]:
gsc_metrics_pattern = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE gsc_impressions IS NULL
        ) AS missing_impressions,

        COUNT(*) FILTER (
            WHERE gsc_clicks IS NULL
        ) AS missing_clicks,

        COUNT(*) FILTER (
            WHERE gsc_sum_position IS NULL
        ) AS missing_sum_position,

        COUNT(*) FILTER (
            WHERE gsc_avg_position IS NULL
        ) AS missing_avg_position
    FROM read_parquet(?)
""", [performance_file]).fetchdf()

gsc_metrics_pattern

,missing_impressions,missing_clicks,missing_sum_position,missing_avg_position
0,0,0,4,7815170


In [79]:
gsc_position_pattern = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE gsc_avg_position IS NULL
        ) AS missing_avg_position,

        COUNT(*) FILTER (
            WHERE gsc_avg_position IS NULL
            AND gsc_impressions = 0
        ) AS missing_with_zero_impressions,

        COUNT(*) FILTER (
            WHERE gsc_avg_position IS NULL
            AND gsc_impressions > 0
        ) AS missing_with_impressions
    FROM read_parquet(?)
""", [performance_file]).fetchdf()

gsc_position_pattern

,missing_avg_position,missing_with_zero_impressions,missing_with_impressions
0,7815170,7815135,35


In [80]:
gsc_position_edge_cases = con.execute("""
    SELECT
        COUNT(*) AS edge_case_count,
        MIN(gsc_impressions) AS min_impressions,
        MAX(gsc_impressions) AS max_impressions,
        AVG(gsc_impressions) AS avg_impressions,
        COUNT(*) FILTER (
            WHERE gsc_sum_position IS NULL
        ) AS missing_sum_position
    FROM read_parquet(?)
    WHERE gsc_avg_position IS NULL
      AND gsc_impressions > 0
""", [performance_file]).fetchdf()

gsc_position_edge_cases

,edge_case_count,min_impressions,max_impressions,avg_impressions,missing_sum_position
0,35,11,269,37.114286,4


In [81]:
gsc_sum_position_pattern = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE gsc_sum_position IS NULL
        ) AS missing_sum_position,

        COUNT(*) FILTER (
            WHERE gsc_sum_position IS NULL
            AND gsc_impressions = 0
        ) AS missing_with_zero_impressions,

        COUNT(*) FILTER (
            WHERE gsc_sum_position IS NULL
            AND gsc_impressions > 0
        ) AS missing_with_impressions
    FROM read_parquet(?)
""", [performance_file]).fetchdf()

gsc_sum_position_pattern

,missing_sum_position,missing_with_zero_impressions,missing_with_impressions
0,4,0,4


# Missing Values Progress

So far, we understand the missing values in `dim_content` and `fact_content_daily_performance` as follows:

## dim_content

* **Main IDs** (`client_hash_id`, `content_hash_id`): No missing values.
* `keyword_hash_id` and `keyword_created_date`: Missing together in `71,998` rows.
* `char_count` and `word_count`: Missing together in `177,768` rows.
* `last_optimized_date` and `optimization_eligible_date`: Missing together in `474,210` rows.
* `search_volume`, `main_intent`, and other fields: Have some missing values. We need to understand the reason for these missing values before deciding how to handle them.

## fact_content_daily_performance

* `gsc_data_available`: No missing values.
* `ga4_data_available`: `2,397,428` missing values.
* When `ga4_data_available` is missing, the related `GA4` and session metrics are also missing.


* `gsc_impressions` and `gsc_clicks`: No missing values.
* `gsc_avg_position`: `7,815,170` missing values.
* `7,815,135` of them have `impressions` = `0`. This looks *logically expected*.
* Only `35` rows have `impressions` > `0`. These are **edge cases** that need further investigation.


* `gsc_sum_position`: `4` missing values.
* All `4` rows have `impressions` > `0`. These are **edge cases** that need further investigation.

In [82]:
query_missing_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE query_hash_id IS NULL
        ) AS missing_query_hash_id,

        COUNT(*) FILTER (
            WHERE window_start IS NULL
        ) AS missing_window_start,

        COUNT(*) FILTER (
            WHERE window_end IS NULL
        ) AS missing_window_end,

        COUNT(*) FILTER (
            WHERE impressions_90d IS NULL
        ) AS missing_impressions_90d,

        COUNT(*) FILTER (
            WHERE clicks_90d IS NULL
        ) AS missing_clicks_90d,

        COUNT(*) FILTER (
            WHERE avg_position_90d IS NULL
        ) AS missing_avg_position_90d
    FROM read_parquet(?)
""", [query_file]).fetchdf()

query_missing_check

,total_rows,missing_query_hash_id,missing_window_start,missing_window_end,missing_impressions_90d,missing_clicks_90d,missing_avg_position_90d
0,2414248,0,0,0,0,0,0


In [83]:
duplicate_check = con.execute("""
    SELECT
        COUNT(*) AS duplicate_rows,

        COUNT(DISTINCT
            md5(
                concat_ws(
                    '|',
                    CAST(report_date AS VARCHAR),
                    client_hash_id,
                    content_hash_id,
                    CAST(client_has_gsc AS VARCHAR),
                    CAST(client_has_ga4 AS VARCHAR),
                    CAST(gsc_data_available AS VARCHAR),
                    CAST(ga4_data_available AS VARCHAR),
                    CAST(gsc_impressions AS VARCHAR),
                    CAST(gsc_clicks AS VARCHAR),
                    CAST(gsc_sum_position AS VARCHAR),
                    CAST(gsc_avg_position AS VARCHAR),
                    CAST(ga4_pageviews AS VARCHAR),
                    CAST(ga4_sessions AS VARCHAR),
                    CAST(ga4_users AS VARCHAR),
                    CAST(ga4_engaged_sessions AS VARCHAR),
                    CAST(ga4_total_engagement_sec AS VARCHAR),
                    CAST(sessions_organic AS VARCHAR),
                    CAST(sessions_direct AS VARCHAR),
                    CAST(sessions_referral AS VARCHAR),
                    CAST(sessions_social AS VARCHAR),
                    CAST(sessions_paid AS VARCHAR),
                    CAST(sessions_ai AS VARCHAR),
                    CAST(ai_chatgpt AS VARCHAR),
                    CAST(ai_perplexity AS VARCHAR),
                    CAST(ai_gemini AS VARCHAR),
                    CAST(ai_copilot AS VARCHAR),
                    CAST(ai_claude AS VARCHAR),
                    CAST(ai_meta AS VARCHAR),
                    CAST(ai_other AS VARCHAR),
                    CAST(scroll_events AS VARCHAR),
                    month
                )
            )
        ) AS unique_full_rows
    FROM read_parquet(?)
""", [performance_file]).fetchdf()

duplicate_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_rows,unique_full_rows
0,11694072,11687682


In [84]:
duplicate_rate = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(DISTINCT md5(
            concat_ws(
                '|',
                CAST(report_date AS VARCHAR),
                client_hash_id,
                content_hash_id,
                CAST(client_has_gsc AS VARCHAR),
                CAST(client_has_ga4 AS VARCHAR),
                CAST(gsc_data_available AS VARCHAR),
                CAST(ga4_data_available AS VARCHAR),
                CAST(gsc_impressions AS VARCHAR),
                CAST(gsc_clicks AS VARCHAR),
                CAST(gsc_sum_position AS VARCHAR),
                CAST(gsc_avg_position AS VARCHAR),
                CAST(ga4_pageviews AS VARCHAR),
                CAST(ga4_sessions AS VARCHAR),
                CAST(ga4_users AS VARCHAR),
                CAST(ga4_engaged_sessions AS VARCHAR),
                CAST(ga4_total_engagement_sec AS VARCHAR),
                CAST(sessions_organic AS VARCHAR),
                CAST(sessions_direct AS VARCHAR),
                CAST(sessions_referral AS VARCHAR),
                CAST(sessions_social AS VARCHAR),
                CAST(sessions_paid AS VARCHAR),
                CAST(sessions_ai AS VARCHAR),
                CAST(ai_chatgpt AS VARCHAR),
                CAST(ai_perplexity AS VARCHAR),
                CAST(ai_gemini AS VARCHAR),
                CAST(ai_copilot AS VARCHAR),
                CAST(ai_claude AS VARCHAR),
                CAST(ai_meta AS VARCHAR),
                CAST(ai_other AS VARCHAR),
                CAST(scroll_events AS VARCHAR),
                month
            )
        )) AS duplicate_rows,

        ROUND(
            100.0 * (
                COUNT(*) - COUNT(DISTINCT md5(
                    concat_ws(
                        '|',
                        CAST(report_date AS VARCHAR),
                        client_hash_id,
                        content_hash_id
                    )
                ))
            ) / COUNT(*),
            4
        ) AS duplicate_rate_percent

    FROM read_parquet(?)
""", [performance_file]).fetchdf()

duplicate_rate

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,duplicate_rows,duplicate_rate_percent
0,11694072,6390,0.0546


# Duplicate Records

The performance sample contains `6,390` exact duplicate rows, representing approximately `0.0546%` of the dataset.

## Data Preparation

* Since the duplicated records are *exact copies*, they can be **safely removed** during the data preparation stage.

In [85]:
invalid_gsc_values = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE gsc_impressions < 0
        ) AS negative_impressions,

        COUNT(*) FILTER (
            WHERE gsc_clicks < 0
        ) AS negative_clicks,

        COUNT(*) FILTER (
            WHERE gsc_sum_position < 0
        ) AS negative_sum_position,

        COUNT(*) FILTER (
            WHERE gsc_avg_position < 0
        ) AS negative_avg_position,

        COUNT(*) FILTER (
            WHERE gsc_clicks > gsc_impressions
        ) AS clicks_greater_than_impressions
    FROM read_parquet(?)
""", [performance_file]).fetchdf()

invalid_gsc_values

,negative_impressions,negative_clicks,negative_sum_position,negative_avg_position,clicks_greater_than_impressions
0,0,0,0,0,0


In [86]:
invalid_ga4_values = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE ga4_pageviews < 0
        ) AS negative_pageviews,

        COUNT(*) FILTER (
            WHERE ga4_sessions < 0
        ) AS negative_sessions,

        COUNT(*) FILTER (
            WHERE ga4_users < 0
        ) AS negative_users,

        COUNT(*) FILTER (
            WHERE ga4_engaged_sessions < 0
        ) AS negative_engaged_sessions,

        COUNT(*) FILTER (
            WHERE ga4_total_engagement_sec < 0
        ) AS negative_engagement_sec,

        COUNT(*) FILTER (
            WHERE ga4_engaged_sessions > ga4_sessions
        ) AS engaged_sessions_greater_than_sessions
    FROM read_parquet(?)
""", [performance_file]).fetchdf()

invalid_ga4_values

,negative_pageviews,negative_sessions,negative_users,negative_engaged_sessions,negative_engagement_sec,engaged_sessions_greater_than_sessions
0,0,0,0,0,0,0


In [87]:
invalid_query_values = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE impressions_90d < 0
        ) AS negative_impressions,

        COUNT(*) FILTER (
            WHERE clicks_90d < 0
        ) AS negative_clicks,

        COUNT(*) FILTER (
            WHERE clicks_90d > impressions_90d
        ) AS clicks_greater_than_impressions,

        COUNT(*) FILTER (
            WHERE avg_position_90d < 0
        ) AS negative_avg_position,

        COUNT(*) FILTER (
            WHERE avg_position_last30 < 0
        ) AS negative_avg_position_last30,

        COUNT(*) FILTER (
            WHERE avg_position_prev30 < 0
        ) AS negative_avg_position_prev30,

        COUNT(*) FILTER (
            WHERE rare_impressions_share < 0
              OR rare_impressions_share > 1
        ) AS invalid_rare_impressions_share,

        COUNT(*) FILTER (
            WHERE anonymized_impressions_share < 0
              OR anonymized_impressions_share > 1
        ) AS invalid_anonymized_impressions_share
    FROM read_parquet(?)
""", [query_file]).fetchdf()

invalid_query_values

,negative_impressions,negative_clicks,clicks_greater_than_impressions,negative_avg_position,negative_avg_position_last30,negative_avg_position_prev30,invalid_rare_impressions_share,invalid_anonymized_impressions_share
0,0,0,0,0,0,0,0,0


In [88]:
invalid_content_values = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE keyword_char_count < 0
        ) AS negative_keyword_char_count,

        COUNT(*) FILTER (
            WHERE keyword_token_count < 0
        ) AS negative_keyword_token_count,

        COUNT(*) FILTER (
            WHERE url_char_count < 0
        ) AS negative_url_char_count,

        COUNT(*) FILTER (
            WHERE search_volume < 0
        ) AS negative_search_volume,

        COUNT(*) FILTER (
            WHERE competition < 0
        ) AS negative_competition,

        COUNT(*) FILTER (
            WHERE cpc < 0
        ) AS negative_cpc,

        COUNT(*) FILTER (
            WHERE backlinks < 0
        ) AS negative_backlinks,

        COUNT(*) FILTER (
            WHERE category_count < 0
        ) AS negative_category_count,

        COUNT(*) FILTER (
            WHERE char_count < 0
        ) AS negative_char_count,

        COUNT(*) FILTER (
            WHERE word_count < 0
        ) AS negative_word_count
    FROM read_parquet(?)
""", [content_file]).fetchdf()

invalid_content_values

,negative_keyword_char_count,negative_keyword_token_count,negative_url_char_count,negative_search_volume,negative_competition,negative_cpc,negative_backlinks,negative_category_count,negative_char_count,negative_word_count
0,0,0,0,0,0,0,0,0,0,0


# Task 3.2 — Duplicates & Invalid Records

Here is a summary of what we discovered during the data quality checks.

## Duplicates

In `fact_content_daily_performance`:

* We found `6,390` duplicate rows.
* They represent `0.0546%` of the sample.
* The duplicates are *exact duplicates*, meaning the repeated rows contain the same values.
* There are no differences in the metrics between the duplicated rows.

### Decision

* We will **remove** these duplicate records later when building the analytical dataset.
* We will **not remove** them from the raw data, so the raw data remains unchanged and traceable.

## Invalid Values

We checked the following three tables:

* `fact_content_daily_performance`
* `fact_content_query_90d`
* `dim_content`

The next step is to identify and document any invalid values found in these tables and decide how they should be handled during the analytical dataset preparation.

In [89]:
date_consistency = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE content_updated_date < content_created_date
        ) AS updated_before_created,

        COUNT(*) FILTER (
            WHERE keyword_created_date > content_updated_date
        ) AS keyword_after_content_update,

        COUNT(*) FILTER (
            WHERE last_optimized_date > content_updated_date
        ) AS optimized_after_content_update,

        COUNT(*) FILTER (
            WHERE optimization_eligible_date > last_optimized_date
        ) AS eligible_after_optimization
    FROM read_parquet(?)
""", [content_file]).fetchdf()

date_consistency

,updated_before_created,keyword_after_content_update,optimized_after_content_update,eligible_after_optimization
0,0,0,0,45396


In [90]:
optimization_date_gap = con.execute("""
    SELECT
        COUNT(*) AS record_count,
        MIN(
            date_diff(
                'day',
                last_optimized_date,
                optimization_eligible_date
            )
        ) AS min_days,
        MAX(
            date_diff(
                'day',
                last_optimized_date,
                optimization_eligible_date
            )
        ) AS max_days,
        AVG(
            date_diff(
                'day',
                last_optimized_date,
                optimization_eligible_date
            )
        ) AS avg_days
    FROM read_parquet(?)
    WHERE optimization_eligible_date > last_optimized_date
""", [content_file]).fetchdf()

optimization_date_gap

,record_count,min_days,max_days,avg_days
0,45396,45,45,45.0


# Data Quality Summary

## Key Findings

### Missing Values

The dataset contains several structured missing-value patterns rather than purely random missingness.

- Keyword metadata is missing for 71,998 content records. `keyword_hash_id` and `keyword_created_date` are missing together.
- Text metadata is missing for 177,768 records. `char_count` and `word_count` are missing together.
- Optimization metadata is missing for 474,210 records. `last_optimized_date` and `optimization_eligible_date` are missing together.
- GA4 availability is missing for 2,397,428 performance records, and the related GA4/session metrics are missing in the same records.
- `gsc_avg_position` is missing for 7,815,170 performance records. Almost all of these records have zero GSC impressions, making the missing position value expected rather than an ordinary data error.
- Only 35 records have missing `gsc_avg_position` despite having impressions.
- The query-level table has no missing values in the key columns examined.

### Duplicates

The performance sample contains 6,390 exact duplicate rows, representing approximately 0.0546% of the sample.

These duplicates are exact copies and will be removed during analytical dataset preparation rather than modifying the raw source data.

### Invalid Values

No negative or logically invalid values were found in the main numeric metrics examined across the content, performance, and query tables.

### Date Consistency

No records were found where content update dates preceded content creation dates.

A total of 45,396 records have `optimization_eligible_date` exactly 45 days after `last_optimized_date`. Because this pattern is completely consistent, it is treated as a dataset/business rule rather than an invalid date relationship.

## Data Quality Decisions

| Issue | Decision |
|---|---|
| Exact duplicate performance rows | Remove during analytical dataset preparation |
| GA4 metrics with unavailable GA4 data | Preserve as missing; do not automatically replace with zero |
| GSC average position with zero impressions | Preserve as missing because position is not meaningful without impressions |
| Rare GSC position edge cases | Document and investigate if relevant to the selected research question |
| Missing content metadata | Preserve initially; treatment depends on the selected research question and features |
| 45-day optimization date pattern | Keep unchanged |
| Invalid negative numeric values | None identified |

## Overall Assessment

The dataset is generally structurally consistent and contains no major widespread numeric validity problems in the fields examined.

Most missing values appear to follow meaningful patterns related to data availability or metadata availability. Therefore, missing values should not be treated as ordinary errors or blindly imputed before the research question and analytical features are defined.